<a href="https://colab.research.google.com/github/CoolingVerseOracle/Coolingverse-data/blob/main/02_Apartment_Preprocessing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1. 파일 로드

## 드라이브에서 파일 로드 허가

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 아파트 위경도 변환 지오코딩
데이터만 바꿔서 바로 적용 가능.
데이터 출처 [링크 텍스트](https://www.k-apt.go.kr/web/board/webReference/boardList.do#

> **⚠️ 실행 전 필독 사항 (Important Notes)**
> **카카오 API 키 입력 필수:** 아래 코드 셀의 `KAKAO_API_KEY = "본인의_REST_API_KEY"` 칸에 개인 카카오 디벨로퍼스 키를 반드시 입력해 주세요!

In [ ]:
import time
import requests
import pandas as pd

# 1. 아파트 엑셀 파일 로드
# (코랩에 업로드하신 파일명을 아래 경로에 맞춰주세요)
filepath = '/content/drive/MyDrive/경기도 부천시 아파트.xlsx'
df = pd.read_excel(filepath)

# 카카오 로컬 API 키 세팅
KAKAO_API_KEY = 'YOUR_KAKAO_API_KEY'
headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}

# 카카오 주소 검색 API 호출기
def get_coordinates(address):
    if pd.isna(address) or not str(address).strip():
        return None, None

    # 카카오 주소 검색 API Endpoint
    url = f'https://dapi.kakao.com/v2/local/search/address.json?query={address}'
    try:
        res = requests.get(url, headers=headers)
        if res.status_code == 200:
            documents = res.json().get('documents', [])
            if documents:
                # y가 위도(latitude), x가 경도(longitude)
                return float(documents[0]['y']), float(documents[0]['x'])
    except Exception as e:
        pass
    return None, None

# 2. 좌표 변환 작업 시작
print(f"🏢 총 {len(df)}개 아파트 단지 좌표 변환을 시작합니다...")

latitudes = []
longitudes = []

for idx, row in df.iterrows():
    # 1순위: 도로명주소로 검색
    lat, lon = get_coordinates(row['도로명주소'])

    # 2순위: 도로명주소 실패 시 법정동주소로 재검색
    if not lat:
        lat, lon = get_coordinates(row['법정동주소'])

    latitudes.append(lat)
    longitudes.append(lon)

    # 과도한 API 트래픽 방지를 위해 미세한 대기
    time.sleep(0.02)

    if (idx + 1) % 50 == 0:
        print(f"  > {idx + 1}개 단지 완료...")

# 3. 데이터프레임에 좌표 컬럼 추가
df['latitude'] = latitudes
df['longitude'] = longitudes

# 결과 리포트 출력
success_count = df['latitude'].notna().sum()
success_rate = (success_count / len(df)) * 100

print("\n" + "="*50)
print(f"📊 [아파트 좌표 변환 최종 리포트]")
print(f"✓ 전체 아파트 단지: {len(df)}개")
print(f"✓ 매칭 성공 단지: {success_count}개")
print(f"🎯 매칭 성공률: {success_rate:.2f}%")
print("="*50)

# 4. 파일 저장 (한글 깨짐 방지를 위해 utf-8-sig 인코딩 적용)
output_path = '/content/drive/MyDrive/경기도_부천시_아파트_좌표포함.csv'
df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"🎉 좌표 추가 완료! 결과 파일이 아래 경로에 저장되었습니다.\n👉 {output_path}")

🏢 총 259개 아파트 단지 좌표 변환을 시작합니다...
  > 50개 단지 완료...
  > 100개 단지 완료...
  > 150개 단지 완료...
  > 200개 단지 완료...
  > 250개 단지 완료...

📊 [아파트 좌표 변환 최종 리포트]
✓ 전체 아파트 단지: 259개
✓ 매칭 성공 단지: 253개
🎯 매칭 성공률: 97.68%
🎉 좌표 추가 완료! 결과 파일이 아래 경로에 저장되었습니다.
👉 /content/drive/MyDrive/경기도_부천시_아파트_좌표포함.csv


## 성남시 누락데이터 구제 코드
성남시의 누락된 데이터 기반 구제 코드

In [ ]:
import re
import time
import requests
import pandas as pd

# 1. 아파트 파일 로드
filepath = '/content/drive/MyDrive/성남시_분당구_아파트_좌표포함.csv'
try:
    df = pd.read_csv(filepath, encoding='utf-8')
except:
    df = pd.read_csv(filepath, encoding='cp949')

# API 키 및 헤더 설정
KAKAO_API_KEY = 'YOUR_KAKAO_API_KEY'
headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}

# 카카오 주소 검색 API 호출 함수
def call_kakao_address(query):
    url = f'https://dapi.kakao.com/v2/local/search/address.json?query={query}'
    try:
        res = requests.get(url, headers=headers)
        if res.status_code == 200:
            docs = res.json().get('documents', [])
            if docs:
                return float(docs[0]['y']), float(docs[0]['x'])
    except:
        pass
    return None, None

# 콤마로 얽힌 주소를 하나로 자르고 단지명 노이즈를 없애는 정제기
def clean_apartment_address(addr):
    if pd.isna(addr) or not str(addr).strip():
        return None

    addr = str(addr).strip()

    # 1단계: 콤마(,)로 여러 주소가 나열된 경우 첫 번째 주소만 선택
    addr = addr.split(',')[0].strip()

    # 2단계: '성남분당구' -> '성남시 분당구'로 통일 보정
    addr = addr.replace("성남분당구", "성남시 분당구")

    # 3단계: 정규표현식으로 동/로/길 번호까지만 정확히 남기고 단지명이나 지저분한 꼬리표 잘라내기
    # (예: "대장동 3- 힐스테이트판교엘포레3블럭" -> "대장동 3-")
    match = re.search(r'(경기도\s+성남시\s+분당구\s+[가-힣\d]+(동|로|길|번길)\s+\d+([-\d]+)?)', addr)
    if match:
        clean = match.group(1).strip()
        # 끝에 남은 불필요한 대시(-) 제거 (예: "서현동 322-" -> "서현동 322")
        if clean.endswith('-'):
            clean = clean[:-1].strip()
        return clean

    return addr

# ==========================================
# 2. 누락 데이터 타겟 정밀 구제 시작
# ==========================================
null_mask = df['latitude'].isna()
failed_rows = df[null_mask]

print(f"🔄 누락된 {len(failed_rows)}개 단지의 정밀 복구를 시작합니다...")

for idx, row in failed_rows.iterrows():
    lat, lon = None, None

    # 1. 도로명주소 우선 정제 후 검색
    cleaned_road = clean_apartment_address(row['도로명주소'])
    if cleaned_road:
        lat, lon = call_kakao_address(cleaned_road)

    # 2. 실패 시 법정동주소 정제 후 검색
    if not lat:
        cleaned_legal = clean_apartment_address(row['법정동주소'])
        if cleaned_legal:
            lat, lon = call_kakao_address(cleaned_legal)

    # 3. 좌표 업데이트
    if lat and lon:
        df.at[idx, 'latitude'] = lat
        df.at[idx, 'longitude'] = lon
        print(f"  ▶ [성공] {row['단지명']} -> ({lat}, {lon})")
    else:
        print(f"  ❌ [실패] {row['단지명']} (주소 검토 필요)")

    time.sleep(0.05)

# 최종 결과 도출
final_nulls = df['latitude'].isna().sum()
print("\n" + "="*50)
print(f"📊 [정밀 구제 완료 최종 리포트]")
print(f"✓ 전체 아파트 수: {len(df)}개")
print(f"✓ 남은 누락 개수: {final_nulls}개")
print(f"🎯 최종 매칭률: {((len(df) - final_nulls)/len(df))*100:.2f}%")
print("="*50)

# 최종 파일 저장
output_path = '/content/drive/MyDrive/성남시_분당구_아파트_좌표포함_최종.csv'
df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"🎉 누락 복구가 완료된 무결점 파일이 저장되었습니다!\n경로: {output_path}")

🔄 누락된 13개 단지의 정밀 복구를 시작합니다...
  ▶ [성공] 분당장안타운건영2차 -> (37.3706450074757, 127.139545493411)
  ▶ [성공] 수내양지마을한양2단지 -> (37.3758089733508, 127.114444116403)
  ▶ [성공] 수내양지마을한양2단지(603동) -> (37.3758089733508, 127.114444116403)
  ▶ [성공] 수내푸른마을신성벽산쌍용 -> (37.3713015752089, 127.125904010586)
  ▶ [성공] 양지마을 금호 1단지 아파트 -> (37.3765713670859, 127.116253996544)
  ▶ [성공] 동양정자파라곤 -> (37.3703245532635, 127.106502566676)
  ▶ [성공] 아이파크분당 -> (37.3712532366232, 127.105602179014)
  ▶ [성공] 서현효자촌동아 -> (37.3781284683827, 127.135436922738)
  ▶ [성공] 서현효자촌삼환 -> (37.3738556730836, 127.133337656875)
  ▶ [성공] 효자촌 정도빌라 -> (37.3752470711015, 127.139859757062)
  ▶ [성공] 무지개마을동아 -> (37.3406877578594, 127.12364926065)
  ▶ [성공] 판교 해링턴 플레이스 -> (37.3690932148454, 127.073016822355)
  ❌ [실패] 힐스테이트판교엘포레3블럭 (주소 검토 필요)

📊 [정밀 구제 완료 최종 리포트]
✓ 전체 아파트 수: 211개
✓ 남은 누락 개수: 1개
🎯 최종 매칭률: 99.53%
🎉 누락 복구가 완료된 무결점 파일이 저장되었습니다!
경로: /content/drive/MyDrive/성남시_분당구_아파트_좌표포함_최종.csv


## 부천시 누락데이터 구제 코드
부천시 누락데이터 맞춤형 구제코드

In [ ]:
import pandas as pd
import requests
import re
import time

# 1. 기존에 저장한 파일 불러오기
output_path = '/content/drive/MyDrive/경기도_부천시_아파트_좌표포함.csv'
df = pd.read_csv(output_path)

# 카카오 API 세팅
KAKAO_API_KEY = 'YOUR_KAKAO_API_KEY'
headers = {"Authorization": f"KakaoAK {KAKAO_API_KEY}"}

def get_coordinates(query, endpoint="address.json"):
    url = f'https://dapi.kakao.com/v2/local/search/{endpoint}?query={query}'
    try:
        res = requests.get(url, headers=headers)
        if res.status_code == 200:
            docs = res.json().get('documents', [])
            if docs:
                return float(docs[0]['y']), float(docs[0]['x'])
    except Exception as e:
        pass
    return None, None

# 🌟 정규식을 활용한 강력한 주소 정제 함수
def clean_address(address):
    if pd.isna(address) or not str(address).strip():
        return ""

    # a. 여러 주소가 콤마(,)로 묶인 경우 첫 번째 주소만 추출
    addr = str(address).split(',')[0].strip()

    # b. 누락된 행정구역(시) 추가 및 교정
    addr = addr.replace("부천원미구", "부천시 원미구")
    addr = addr.replace("부천소사구", "부천시 소사구")
    addr = addr.replace("부천오정구", "부천시 오정구")

    # c. 끝에 의미 없이 붙은 하이픈(-) 및 텍스트 제거 (예: "14- 아파트" -> "14", "1152-" -> "1152")
    addr = re.sub(r'-\s*\D*$', '', addr)

    # d. 번지수 뒤에 띄어쓰기로 아파트명이 붙은 경우 번지수까지만 추출 (예: "중동 1152 상록타워" -> "중동 1152")
    match = re.search(r'(.*?\d+(?:-\d+)?)', addr)
    if match:
        addr = match.group(1).strip()

    return addr

# 2. 누락된 데이터(결측치)만 추출
missing_indices = df[df['latitude'].isna()].index
print(f"🛠 전체 {len(df)}개 중 누락된 {len(missing_indices)}개 단지에 대해 긴급 복구를 시작합니다...\n")

recovered_count = 0

for idx in missing_indices:
    row = df.loc[idx]

    # 도로명주소, 법정동주소 노이즈 제거
    c_road = clean_address(row['도로명주소'])
    c_jibun = clean_address(row['법정동주소'])

    lat, lon = None, None

    # 1순위: 클렌징된 도로명주소
    if c_road:
        lat, lon = get_coordinates(c_road)

    # 2순위: 클렌징된 법정동주소
    if not lat and c_jibun:
        lat, lon = get_coordinates(c_jibun)

    # 3순위: 그래도 안 되면 '키워드 장소 검색(keyword.json)' API로 직접 타격
    if not lat:
        keyword = f"부천 {row['단지명']}"
        lat, lon = get_coordinates(keyword, endpoint="keyword.json")

    # 결과 업데이트
    if lat and lon:
        df.at[idx, 'latitude'] = lat
        df.at[idx, 'longitude'] = lon
        recovered_count += 1
        print(f"✅ [복구 성공] {row['단지명']} (인식된 주소: {c_road if c_road else c_jibun})")
    else:
        print(f"❌ [복구 실패] {row['단지명']} (수동 확인 필요)")

    time.sleep(0.05)

# 3. 최종 결과 저장
final_success = df['latitude'].notna().sum()
print("\n" + "="*50)
print(f"🎯 [누락 데이터 복구 완료]")
print(f"✓ 복구 성공: {recovered_count}개")
print(f"✓ 최종 매칭 성공률: {(final_success/len(df))*100:.2f}% ({final_success}/{len(df)}개)")
print("="*50)

# 원본 파일에 덮어쓰기
df.to_csv(output_path, index=False, encoding='utf-8-sig')
print("💾 결과 파일 덮어쓰기 저장 완료!")

🛠 전체 259개 중 누락된 6개 단지에 대해 긴급 복구를 시작합니다...

✅ [복구 성공] 약대현대아파트 (인식된 주소: 경기도 부천시 원미구 약대동 169-6)
✅ [복구 성공] 상록센트럴타워 (인식된 주소: 경기도 부천시 원미구 중동 1152)
✅ [복구 성공] 소새울역신일해피트리아파트 (인식된 주소: 경기도 부천시 소사구 소사본동 427)
✅ [복구 성공] 덕림현대아파트 (인식된 주소: 경기도 부천시 소사구 괴안동 14)
✅ [복구 성공] 현대허니문아파트 (인식된 주소: 경기도 부천시 소사구 괴안동 15)
✅ [복구 성공] 부천삼익125동 (인식된 주소: 경기도 부천시 소사구 경인로134)

🎯 [누락 데이터 복구 완료]
✓ 복구 성공: 6개
✓ 최종 매칭 성공률: 100.00% (259/259개)
💾 결과 파일 덮어쓰기 저장 완료!


## 아파트 테이블 그리드 반영 및 er 다이어그램 맞게 정의

In [ ]:
import pandas as pd
import numpy as np
from scipy.spatial import cKDTree

# 1. 원본 데이터 로드 (안전한 인코딩 처리)
def load_csv_safe(path):
    encodings = ['utf-8-sig', 'cp949', 'euc-kr', 'utf-8']
    for enc in encodings:
        try:
            return pd.read_csv(path, encoding=enc)
        except:
            pass
    return pd.read_csv(path)

# 파일 경로 (코랩 환경에 맞게 수정)
apt_path = '/content/drive/MyDrive/경기도_부천시_아파트_좌표포함.csv'
grid_path = '/content/drive/MyDrive/grids_bucheon_final.csv'

df_apt = load_csv_safe(apt_path)
df_grid = load_csv_safe(grid_path)

print(f"🏢 아파트 데이터: {len(df_apt)}건, 격자 데이터: {len(df_grid)}건 로드 완료")

# 2. 아파트 좌표와 가장 가까운 Grid ID 자동 매핑 (KDTree 알고리즘 사용)
# 결측치 방어 (혹시라도 좌표가 없는 경우 0 처리)
df_apt['latitude'] = df_apt['latitude'].fillna(0)
df_apt['longitude'] = df_apt['longitude'].fillna(0)

# 그리드 중심 좌표와 아파트 좌표 배열화
grid_coords = df_grid[['center_lat', 'center_lng']].values
apt_coords = df_apt[['latitude', 'longitude']].values

# KD-Tree 구성 및 가장 가까운 거리의 인덱스 탐색
tree = cKDTree(grid_coords)
distances, indices = tree.query(apt_coords)

# 찾은 인덱스를 바탕으로 grid_id 맵핑
df_apt['grid_id'] = df_grid.iloc[indices]['grid_id'].values

# 3. ERD 명세에 맞춘 최종 데이터프레임 재구성
df_transformed = pd.DataFrame()

df_transformed['apartment_id'] = range(1, len(df_apt) + 1)  # PK 자동 부여
df_transformed['grid_id'] = df_apt['grid_id']               # 방금 매핑한 격자 ID
df_transformed['kapt_code'] = df_apt['단지코드']
df_transformed['name'] = df_apt['단지명']
df_transformed['address'] = df_apt['법정동주소'].fillna(df_apt['도로명주소'])
df_transformed['lat'] = df_apt['latitude']
df_transformed['lng'] = df_apt['longitude']
df_transformed['total_parking'] = df_apt['총주차대수'].fillna(0).astype(int)

# [외부인 개방 여부 Y/N 판단 로직]
def convert_is_open(row):
    ground = str(row['외부인개방여부(지상)']) if pd.notna(row['외부인개방여부(지상)']) else ''
    underground = str(row['외부인개방여부(지하)']) if pd.notna(row['외부인개방여부(지하)']) else ''
    open_keywords = ['허용', '개방', 'Y']

    # 지상/지하 중 하나라도 키워드 포함 시 Y
    if any(k in ground for k in open_keywords) or any(k in underground for k in open_keywords):
        return 'Y'
    return 'N'

df_transformed['is_open'] = df_apt.apply(convert_is_open, axis=1)

# ★ [핵심] 부천시 특화 평일 낮 통근 이탈률 적용 (78.8% * 38.2% = 약 0.301)
df_transformed['open_count'] = (df_transformed['total_parking'] * 0.301).round().astype(int)
df_transformed['source'] = '부천시 공공데이터'

# 4. 변환된 CSV 저장 (백엔드 적재용 DB 규격)
output_file = '/content/drive/MyDrive/bucheon_apartments_erd.csv'
df_transformed.to_csv(output_file, index=False, encoding='utf-8-sig')

print(f"\n🎉 교정 완벽 성공! 모든 아파트가 격자에 매핑되었습니다.")
print(f"👉 파일 저장 위치: {output_file}")

🏢 아파트 데이터: 259건, 격자 데이터: 55256건 로드 완료

🎉 교정 완벽 성공! 모든 아파트가 격자에 매핑되었습니다.
👉 파일 저장 위치: /content/drive/MyDrive/bucheon_apartments_erd.csv
